### Run Scraping Function from National Pokedex Notebook

In [ ]:
%run 'National Pokedex.ipynb'

In [ ]:
import pickle
from IPython.display import display, clear_output
import warnings
warnings.simplefilter("ignore", DeprecationWarning)
# Maximize display of all dataframes
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

### Scrape and Apply Transformations to Pokémon Natures Table from Serebii.net Online Database

In [ ]:
natures_url = 'https://www.serebii.net/games/natures.shtml'
scrape_table(natures_url)
natures = df
# Apply transformations
natures.columns = natures.iloc[0]
natures = natures.iloc[1:]
natures = natures.rename(columns = {'Natures Name':'Nature', 'Raises':'Raised Stat', 'Lowers':'Lowered Stat', 
                                    'Loved Pokéblock':'Favorite Poffin', 
                                    'Hated Pokéblock':'Least Favorite Poffin'}).drop(['Berry','Gen 1 Remainder'], axis=1)
columns = ['Raised Stat', 'Lowered Stat', 'Favorite Poffin', 'Least Favorite Poffin']
for column in columns:
    natures[column] = natures[column].str.replace('None', 'Neutral')
natures['Nature'] = natures['Nature'].str.replace(r'[^\x00-\x7F]+', '', regex=True).str.replace('\t', '')
natures

### Scrape and Apply Transformations to Move Tables Broken Down by Type

#### Looping through each table and storing it in a dictionary is efficient if all data is needed at once or for data analysis (efficient and flexible, however can be cost intentive)

#### FireRed/LeafGreen, Ruby/Sapphire/Emerald - Gen3

In [ ]:
gen3_move_types = ['bug','dark','dragon','electric','fighting','fire','flying','ghost','grass','ground','ice','normal','poison','psychict','rock','water','steel']
gen3_moves = {}
for move_type in gen3_move_types:
    gen3_moves_url=f'https://www.serebii.net/attackdex/{move_type}.shtml'
    scrape_table(gen3_moves_url)
    # Apply transformations
    df.columns = df.iloc[6].str.strip()
    df = df.iloc[7:].rename(columns = {'Att.':'Power', 'Acc.':'Accuracy'})
    df = df[['Name', 'PP', 'Power', 'Accuracy', 'Effect']]
    df = df[~df['Name'].str.contains('All Content')]
    df['Power'] = df['Power'].replace('--', pd.NA).fillna(0).astype('int')  # Reformat moves with no damage output to allow for sorting
    df = df.sort_values('Power', ascending=True)
    df['Power'] = df['Power'].replace(0, '--')
    gen3_moves[move_type] = df
gen3_moves['psychic'] = gen3_moves.pop('psychict')

In [ ]:
# Call move tables by type from dictionary
gen3_moves['dragon']

#### Diamond/Pearl/Platinum, HeartGold/SoulSilver - Gen4

##### Since physical and special attacks for each type were first introduced in gen4, I am adding an additional column to specify such

In [ ]:
gen4_move_types = ['bug','dark','dragon','electric','fighting','fire','flying','ghost','grass','ground','ice','normal','poison','psychict','rock','water','steel']
gen4_moves = {}
for move_type in gen4_move_types:
    gen4_moves_url=f'https://www.serebii.net/attackdex-dp/{move_type}.shtml'
    scrape_table(gen4_moves_url)
    # Apply transformations
    df.columns = df.iloc[4].str.strip()
    df = df.iloc[5:].rename(columns = {'Cat.':'Category', 'Att.':'Power', 'Acc.':'Accuracy'})
    df = df[['Name', 'Category', 'PP', 'Power', 'Accuracy', 'Effect']]
    df = df[~df['Name'].str.contains('All Content')]
    df['Category'] = df['Category'].replace('https://www.serebii.net/pokedex-dp/type/physical.png', 'Physical')\
                                .replace('https://www.serebii.net/pokedex-dp/type/special.png', 'Special')\
                                .replace('https://www.serebii.net/pokedex-dp/type/other.png', 'Other') 
    df['Power'] = df['Power'].replace('--', pd.NA).fillna(0).astype('int')  # Reformat moves with no damage output to allow for sorting
    df = df.sort_values('Power', ascending=True)
    df['Power'] = df['Power'].replace(0, '--')
    gen4_moves[move_type] = df
gen4_moves['psychic'] = gen4_moves.pop('psychict')

In [ ]:
# Call gen4 move tables by type
gen4_moves['psychic']

#### Black/White, Black/White 2 - Gen5

In [ ]:
gen5_move_types = ['bug','dark','dragon','electric','fighting','fire','flying','ghost','grass','ground','ice','normal','poison','psychict','rock','water','steel']
gen5_moves = {}
for move_type in gen5_move_types:
    gen5_moves_url=f'https://www.serebii.net/attackdex-bw/{move_type}.shtml'
    scrape_table(gen5_moves_url)
    # Apply transformations
    df.columns = df.iloc[3].str.strip()
    df = df.iloc[4:].rename(columns = {'Cat.':'Category', 'Att.':'Power', 'Acc.':'Accuracy'})
    df = df[['Name', 'PP', 'Category', 'Power', 'Accuracy', 'Effect']]
    df = df[~df['Name'].str.contains('All Content')]
    df['Category'] = df['Category'].replace('https://www.serebii.net/pokedex-dp/type/physical.png', 'Physical')\
                                .replace('https://www.serebii.net/pokedex-dp/type/special.png', 'Special')\
                                .replace('https://www.serebii.net/pokedex-dp/type/other.png', 'Other')
    df['Power'] = df['Power'].replace('--', pd.NA).fillna(0).astype('int')  # Reformat moves with no damage output to allow for sorting
    df = df.sort_values('Power', ascending=True)
    df['Power'] = df['Power'].replace(0, '--')
    gen5_moves[move_type] = df
gen5_moves['psychic'] = gen5_moves.pop('psychict')

In [ ]:
# Call gen5 move tables by type
gen5_moves['psychic']

### Scrape and Apply Transformations to Pokemon Moveset Tables for Each Generation

In [ ]:
pokemon_list = pokedex_df['Pokémon'].tolist()
pokemon_name_corrections = {
    # Change names to match urls
    "nidoran♀": "nidoran-f",
    "nidoran♂": "nidoran-m",
    "mr. mime": "mr-mime",
    "mime jr.": "mime-jr",
    "farfetch'd": "farfetchd",
    # Replace normal forms with standard names used in urls
    "deoxys normal forme": "deoxys",
    "burmy plant cloak": "burmy",
    "wormadam plant cloak": "wormadam",
    "shaymin land forme": "shaymin",
    "basculin red-striped form": "basculin",
    "darmanitan standard mode": "darmanitan",
    "tornadus incarnate forme": "tornadus",
    "thundurus incarnate forme": "thundurus",
    "landorus incarnate forme": "landorus",
    "meloetta aria forme": "meloetta",
    "keldeo resolute form": "keldeo"
}
# Apply corrections to pokemon_list
pokemon_list = [pokemon_name_corrections.get(pokemon.lower(), pokemon) for pokemon in pokemon_list]
# Remove alternative forms from list, as they are not scrapable as their own urls
remove_list=['CASTFORM SUNNY FORM','CASTFORM RAINY FORM','CASTFORM SNOWY FORM','DEOXYS ATTACK FORME','DEOXYS DEFENSE FORME',
             'DEOXYS SPEED FORME','BURMY SANDY CLOAK','BURMY TRASH CLOAK','WORMADAM SANDY CLOAK','WORMADAM TRASH CLOAK','ROTOM HEAT FORME',
             'ROTOM WASH FORME','ROTOM FROST FORME','ROTOM FAN FORME','ROTOM MOW FORME','GIRATINA DISTORTION FORME','SHAYMIN SKY FORME',
             'BASCULIN WHITE-STRIPED FORM','BASCULIN BLUE-STRIPED FORM','DARMANITAN ZEN MODE','TORNADUS THERIAN FORME',
             'THUNDURUS THERIAN FORME','LANDORUS THERIAN FORME','MELOETTA PIROUETTE FORME','KYUREM WHITE','KYUREM BLACK',
             'KELDEO RESOLUTE FORM']
for pokemon in remove_list:
    if pokemon in pokemon_list:
        pokemon_list.remove(pokemon)

#### FireRed/LeafGreen, Ruby/Sapphire/Emerald - Gen3

In [ ]:
# This loop scrapes moveset data for hundreds of pokemon. Running this more than once for data analysis is computationally expensive and time-consuming. Therefore, this loop is only run once, then a dictionary containing each moveset dataframe is stored locally using pickle for further data anaylsis.
# Remove "if False" clause to run the loop for the first time
if False:
    gen3_movesets = {}
    for pokemon in pokemon_list:
        gen3_moveset_url=f'https://pokemondb.net/pokedex/{pokemon}/moves/3'
        df = scrape_table(gen3_moveset_url)
        # Break the loop once it reaches gen4 pokemon
        if df is None:
            break
        # Apply transformations
        df.columns = df.iloc[0]
        df = df.iloc[1:].rename(columns = {'Lv.':'Level Learned', 'Acc.':'Accuracy'})
        df = df.drop(['Cat.'], axis=1)
        df=df.loc[:df['Level Learned'].str.contains('Move|HM', na=False).idxmax() - 1]  # Cap table at level up movesets, ignore additional move data such as TMs/HMs
        gen3_movesets[pokemon] = df
    gen3_movesets = {key.lower(): value for key, value in gen3_movesets.items()}  # Convert dictionary keys to lowercase to make querying easier
    # Store gen3 pokemon movesets dictionary as a pkl file 
    with open('gen3_movesets.pkl', 'wb') as a:
        pickle.dump(gen3_movesets, a)

In [ ]:
# Load the gen3 movesets dictionary from pkl file storage. 
# This will allow you to query all previously scraped tables for data analysis without having to run the loop again.
with open('gen3_movesets.pkl', 'rb') as a:
    gen3_movesets = pickle.load(a)

In [ ]:
# Call gen3 pokemon moveset
gen3_movesets['deoxys']

#### Diamond/Pearl/Platinum, HeartGold/SoulSilver - Gen4

In [ ]:
# This loop scrapes moveset data for hundreds of pokemon. Running this more than once for data analysis is computationally expensive and time-consuming. Therefore, this loop is only run once, then a dictionary containing each moveset dataframe is stored locally using pickle for further data anaylsis.
# Remove "if False" clause to run the loop for the first time
if False:
    gen4_movesets = {}
    for pokemon in pokemon_list:
        gen4_moveset_url=f'https://pokemondb.net/pokedex/{pokemon}/moves/4'
        df = scrape_table(gen4_moveset_url)
        # Break the loop once it reaches gen5 pokemon
        if df is None:
            break
        # Apply transformations
        df.columns = df.iloc[0]
        df = df.iloc[1:].rename(columns = {'Lv.':'Level Learned', 'Cat.':'Category', 'Att.':'Power', 'Acc':'Accuracy'})
        df['Category'] = df['Category'].replace('https://www.serebii.nethttps://img.pokemondb.net/images/icons/move-physical.png', 'Physical')\
                                .replace('https://www.serebii.nethttps://img.pokemondb.net/images/icons/move-special.png', 'Special')\
                                .replace('https://www.serebii.nethttps://img.pokemondb.net/images/icons/move-status.png', 'Other')
        df = df.loc[:df['Level Learned'].str.contains('Move|HM', na=False).idxmax() -1]  # Cap table at level up movesets, ignore additional move data such as TMs/HMs
        gen4_movesets[pokemon] = df
    gen4_movesets = {key.lower(): value for key, value in gen4_movesets.items()}  # Convert dictionary keys to lowercase to make querying easier
    # Store gen4 pokemon movesets dictionary as a pkl file 
    with open('gen4_movesets.pkl', 'wb') as b:
        pickle.dump(gen4_movesets, b)

In [ ]:
# Load the gen4 movesets dictionary from pkl file storage. 
# This will allow you to query all previously scraped tables for data analysis without having to run the loop again.
with open('gen4_movesets.pkl', 'rb') as b:
    gen4_movesets = pickle.load(b)

In [ ]:
# Call gen4 pokemon moveset
gen4_movesets['arceus']

#### Black/White, Black/White 2 - Gen5

In [ ]:
# This loop scrapes moveset data for hundreds of pokemon. Running this more than once for data analysis is computationally expensive and time-consuming. Therefore, this loop is only run once, then a dictionary containing each moveset dataframe is stored locally using pickle for further data anaylsis.
# Remove "if False" clause to run the loop for the first time
if False:
    gen5_movesets = {}
    for pokemon in pokemon_list:
        gen5_moveset_url=f'https://pokemondb.net/pokedex/{pokemon}/moves/5' 
        df = scrape_table(gen5_moveset_url)
        # Apply transformations
        df.columns = df.iloc[0]
        df = df.iloc[1:].rename(columns = {'Lv.':'Level Learned', 'Cat.':'Category', 'Att.':'Power', 'Acc.':'Accuracy'})
        df['Category'] = df['Category'].replace('https://www.serebii.nethttps://img.pokemondb.net/images/icons/move-physical.png','Physical')\
                                .replace('https://www.serebii.nethttps://img.pokemondb.net/images/icons/move-special.png', 'Special')\
                                .replace('https://www.serebii.nethttps://img.pokemondb.net/images/icons/move-status.png', 'Other')
        df = df.loc[:df['Level Learned'].str.contains('Move|HM', na=False).idxmax() -1]
        gen5_movesets[pokemon] = df
    gen5_movesets = {key.lower(): value for key, value in gen5_movesets.items()}  # Convert dictionary keys to lowercase to make querying easier
    # Store gen5 pokemon movesets dictionary as a pkl file 
    with open('gen5_movesets.pkl', 'wb') as c:
        pickle.dump(gen5_movesets, c)

In [ ]:
# Load the gen5 movesets dictionary from pkl file storage. 
# This will allow you to query all previously scraped tables for data analysis without having to run the loop again.
with open('gen5_movesets.pkl', 'rb') as c:
    gen5_movesets = pickle.load(c)

In [ ]:
# Call gen5 pokemon moveset
gen5_movesets['genesect']

##### Rather than looping through every pokemon and scraping hundreds of tables for each, you can define a function that scrapes moveset data only for pokemon of interest (saves memory/less cost intensive, better for interactive applications, however requires redundant scraping requests and is therefore less efficient if you want to perform data analysis)

In [ ]:
if False:
    def gen3_moveset(pokemon):
        moveset_url=f'https://pokemondb.net/pokedex/{pokemon}/moves/3'
        scrape_table(moveset_url) 
        df.columns=df.iloc[0]
        df=df.iloc[1:].rename(columns={'Lv.':'Level','Power':'Att.'}).drop(['Cat.'], axis=1)
        df=df.loc[:df['Level'].str.contains('Move', na=False).idxmax() - 1]
        return df
    gen3_moveset('treecko')